In [0]:
## libs python

import logging

from pathlib import Path

In [0]:
## libs pyspark

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from delta.tables import DeltaTable


from pyspark.sql.types import DoubleType, LongType
    
    

In [0]:
import warnings
warnings.filterwarnings('ignore')

In [0]:
CATALOG =  'fishing'
RAW  = 'raw'
BRONZE = 'bronze'

FISHING_VOLUME       = f'/Volumes/{CATALOG}/{RAW}/fishing/'

In [0]:
def setup_logging(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
        logger.addHandler(handler)
    
    return logger

In [0]:
LOGGER = setup_logging('Bronze Layer Ingestion')

In [0]:
def read_data(file_path: Path) -> DataFrame:
    return (
        spark
        .read
        .format('parquet')
        .load(str(file_path))
    )


In [0]:
def merge_to_delta(df: DataFrame, table_name: str) -> None:
    # 1. Ensure table exists as a valid Delta table first
    if not spark.catalog.tableExists(table_name):
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .saveAsTable(table_name)
        LOGGER.info(f"Tabela Delta {table_name} criada com sucesso.")
        return

    # 2. Perform upsert if Delta table already exists
    delta_table = DeltaTable.forName(spark, table_name)
    (
        delta_table.alias('target')
        .merge(
            df.alias('source'),
            'target.mmsi = source.mmsi AND target.timestamp = source.timestamp'
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    LOGGER.info(f'MERGE concluído em {table_name}.')

In [0]:
FIXED_GEAR_PARQUET_PATH = Path(FISHING_VOLUME, 'parquet/fishing_fixed_gear_nm_df')
POLE_AND_LINE_PARQUET_PATH = Path(FISHING_VOLUME, 'parquet/fishing_pole_and_line_nm_df')
PURSE_SEINES_PARQUET_PATH = Path(FISHING_VOLUME, 'parquet/fishing_purse_seines_df_nm_df')
FIXED_GEAR_TABLE_NAME = f'{CATALOG}.{BRONZE}.fixed_gear'
POLE_AND_LINE_TABLE_NAME = f'{CATALOG}.{BRONZE}.pole_and_line'
PURSE_SEINES_TABLE_NAME = f'{CATALOG}.{BRONZE}.purse_seines'


In [0]:
LOGGER.info(f'Iniciando ingestão dos dados pesqueiros da camada bronze.')

fixed_gear_df = read_data(FIXED_GEAR_PARQUET_PATH)


fixed_gear_df = (
    fixed_gear_df
    .select(
        F.col('mmsi').alias('mmsi'),
        F.to_date(F.col('timestamp'), 'yyyy-MM-dd').alias('timestamp'),
        F.col('lat').alias('latitude'),
        F.col('lon').alias('longitude'),
        F.col('is_fishing').alias('is_fishing'),
    )
)

display(fixed_gear_df)

# ingestao
merge_to_delta(fixed_gear_df, FIXED_GEAR_TABLE_NAME)

LOGGER.info(f'Ingestão dos dados  da camada bronze concluída com sucesso!')

In [0]:
LOGGER.info(f'Iniciando ingestão dos dados pesqueiros da camada bronze.')

pole_and_line_df = read_data(POLE_AND_LINE_PARQUET_PATH)


pole_and_line_df = (
    pole_and_line_df
    .select(
        F.col('mmsi').alias('mmsi'),
        F.to_date(F.col('timestamp'), 'yyyy-MM-dd').alias('timestamp'),
        F.col('lat').alias('latitude'),
        F.col('lon').alias('longitude'),
        F.col('is_fishing').alias('is_fishing'),
    )
)

display(pole_and_line_df)

# ingestao
merge_to_delta(pole_and_line_df, POLE_AND_LINE_TABLE_NAME)

LOGGER.info(f'Ingestão dos dados  da camada bronze concluída com sucesso!')

In [0]:
LOGGER.info(f'Iniciando ingestão dos dados pesqueiros da camada bronze.')

purse_seines_df = read_data(PURSE_SEINES_PARQUET_PATH)


purse_seines_df = (
    purse_seines_df
    .select(
        F.col('mmsi').alias('mmsi'),
        F.to_date(F.col('timestamp'), 'yyyy-MM-dd').alias('timestamp'),
        F.col('lat').alias('latitude'),
        F.col('lon').alias('longitude'),
        F.col('is_fishing').alias('is_fishing'),
    )
)

display(purse_seines_df)

# ingestao
merge_to_delta(purse_seines_df, PURSE_SEINES_TABLE_NAME)

LOGGER.info(f'Ingestão dos dados  da camada bronze concluída com sucesso!')